# MediScan - Evidence-Grounded Medical RAG Pipeline

This notebook demonstrates the end-to-end architecture of the **MediScan Vector Database (VDB) & Retrieval Engine**.

### Architecture Flow:
```
Source Registry (CSV) ──> Acquisition ──> Cleaning ──> Metadata Enrichment ──> Section-Aware Chunking
                                                                                        │
                              ┌─────────────────────────────────────────────────────────┴─────────────┐
                              ↓                                                                       ↓
                 Dense Vector Index (ChromaDB)                                           Sparse Lexical Index (BM25)
                (NVIDIA NIM 2048-dim Embeddings)                                         (Exact terms / Acronyms)
                              └─────────────────────────────────┬─────────────────────────────────────┘
                                                                ↓
                                                     Hybrid Retrieval Fusion (RRF)
                                                                ↓
                                                     Metadata Routing & Filtering
                                                                ↓
                                                     Evidence Selection & Citations
                                                                ↓
                                                    Downstream MediScan Agent Tools
```

## 1. Imports & Configuration

In [1]:
import sys
from pathlib import Path

# Add src directory to path
sys.path.insert(0, str(Path.cwd()))

import VDB
from VDB.config import VECTORSTORE_DIR, SOURCE_REGISTRY_CSV, EMBEDDING_MODEL
from VDB.schema import MedicalDocument, MedicalChunk, RetrievalFilter, SearchResult
from VDB.pipeline import MediScanRetriever
from VDB.acquisition.registry import load_source_registry
from VDB.acquisition.openi_loader import load_all_openi_reports
from VDB.acquisition.local_loader import load_local_cleaned_documents
from VDB.processing.chunker import SectionAwareChunker
from VDB.indexing.index_builder import build_complete_knowledge_index
from VDB.evaluation.evaluator import RetrievalEvaluator

print(f"MediScan VDB Package loaded successfully!")
print(f"Active Embedding Model: {EMBEDDING_MODEL}")
print(f"ChromaDB Persistence Directory: {VECTORSTORE_DIR}")

MediScan VDB Package loaded successfully!
Active Embedding Model: nvidia/llama-nemotron-embed-vl-1b-v2
ChromaDB Persistence Directory: C:\Users\merna\OneDrive\Desktop\Orange_training_AI_Agents\MediScan\vectorstore\chromadb


## 2. Source Registry & Multi-Source Document Acquisition
- **Clinical Guidelines** (ACR, NICE, AHA, AUA, NCCN, WHO)
- **Clinical References** (StatPearls, NIH Bookshelf)
- **Peer-Reviewed Research** (Systematic Reviews & Meta-Analyses from PMC)
- **Patient Education** (MedlinePlus guides)
- **Radiology Reports & Cases** (OpenI Indiana University CXR Collection & PMC Case Reports)

In [2]:
# 1. Inspect registered sources
registry_sources = load_source_registry()
print(f"Total Registered Knowledge Sources: {len(registry_sources)}\n")

for s in registry_sources[:5]:
    print(f"[{s.get('source_id')}] {s.get('title')} ({s.get('condition')} | {s.get('knowledge_domain')} | {s.get('audience')})")

Total Registered Knowledge Sources: 87

[SRC001] American College of Radiology Appropriateness Criteria for Chest Imaging in Pneumonia (Pneumonia | clinical_guideline | clinician)
[SRC002] NICE Guideline NG95: Pneumonia (adult) management (Pneumonia | clinical_guideline | clinician)
[SRC003] StatPearls: Pneumonia (Pneumonia | clinical_reference | mixed)
[SRC004] Systematic Review of Imaging Modalities for Community‑Acquired Pneumonia (Pneumonia | research | clinician)
[SRC005] MedlinePlus: Pneumonia (Pneumonia | patient_education | patient)


In [3]:
# 2. Load documents with structural metadata
local_docs = load_local_cleaned_documents()
openi_sample_docs = load_all_openi_reports(max_reports=10)

print(f"\nSample Independent OpenI XML Report:")
sample = openi_sample_docs[0]
print(f"  Doc ID: {sample.doc_id}")
print(f"  Condition Tag: {sample.condition}")
print(f"  Sections: {list(sample.sections.keys())}")
print(f"  Findings: {sample.sections.get('FINDINGS', '')[:120]}...")
print(f"  Impression: {sample.sections.get('IMPRESSION', '')}")

Loaded 97 local cleaned reference documents.
Loaded 10 independent OpenI XML reports.

Sample Independent OpenI XML Report:
  Doc ID: OpenI_CXR1
  Condition Tag: Pneumothorax
  Sections: ['INDICATION', 'COMPARISON', 'FINDINGS', 'IMPRESSION']
  Findings: The cardiac silhouette and mediastinum size are within normal limits. There is no pulmonary edema. There is no focal con...
  Impression: Normal chest x-XXXX.


## 3. Section-Aware Chunking with Deterministic Stable Chunk IDs
Preserves clinical boundaries (`[FINDINGS]`, `[IMPRESSION]`, `[DIAGNOSIS]`, `[TREATMENT]`) without cross-document mixing.

In [4]:
chunker = SectionAwareChunker(chunk_size=800, chunk_overlap=150)
sample_chunks = chunker.chunk_document(local_docs[0])

print(f"Document: '{local_docs[0].title}' -> Created {len(sample_chunks)} section-aware chunks\n")
for i, chunk in enumerate(sample_chunks[:3], 1):
    print(f"--- Chunk {i} ---")
    print(f"Chunk ID: {chunk.chunk_id}")
    print(f"Section: {chunk.section_title} | Domain: {chunk.knowledge_domain} | Audience: {chunk.audience}")
    print(f"Content:\n{chunk.content[:200]}...\n")

Document: 'Cardiomegaly' -> Created 52 section-aware chunks

--- Chunk 1 ---
Chunk ID: LOCAL_Cardiomegaly#MAIN_0
Section: MAIN | Domain: clinical_references | Audience: clinician
Content:
[Cardiomegaly]
Cardiomegaly - StatPearls - NCBI Bookshelf
NCBI Bookshelf. A service of the National Library of Medicine, National Institutes of Health.StatPearls . Treasure Island (FL): StatPearls Pub...

--- Chunk 2 ---
Chunk ID: LOCAL_Cardiomegaly#MAIN_1
Section: MAIN | Domain: clinical_references | Audience: clinician
Content:
[Cardiomegaly]
CardiomegalyAuthorsHina Amin1; Waqas J. Siddiqui2.Affiliations1 SUNY Upstate Medical University2 Drexel UniversityLast Update: November 20, 2022.Continuing Education ActivityCardiomegal...

--- Chunk 3 ---
Chunk ID: LOCAL_Cardiomegaly#MAIN_2
Section: MAIN | Domain: clinical_references | Audience: clinician
Content:
[Cardiomegaly]
. Participants learn to recognize risk factors and presenting features, interpret imaging findings, including cardiothoracic ratio on

## 4. Dual Indexing (Dense ChromaDB + Sparse BM25)
- **ChromaDB**: NVIDIA NIM 2048-dim vectors (`nvidia/llama-nemotron-embed-vl-1b-v2`)
- **BM25**: Exact lexical keyword and acronym index for clinical terms

In [5]:
# Connect to persisted ChromaDB and BM25 index
retriever = MediScanRetriever()

print(f"ChromaDB Vector Store Collection: '{retriever.vector_index.collection_name}'")
print(f"Total Chunks in ChromaDB: {retriever.vector_index.count()}")
print(f"BM25 Corpus Size: {len(retriever.bm25_index.chunks)} chunks")

ChromaDB Vector Store Collection: 'mediscan_rag'
Total Chunks in ChromaDB: 2296
BM25 Corpus Size: 0 chunks


## 5. Specialized Agent Tool Bridges
Direct API methods designed for downstream MediScan Agent Tools:

In [6]:
# 1. ClinicalGuidelineTool Bridge
print("=== [A] ClinicalGuidelineTool Bridge ===")
guideline_res = retriever.search_guidelines(
    query="guideline recommendations for acute pneumonia diagnosis and imaging",
    condition="Pneumonia",
    k=2
)
for r in guideline_res:
    print(f"[{r.chunk.source_id}] {r.chunk.title} ({r.chunk.chunk_id}) | Score: {r.score:.4f}")

# 2. SimilarCaseTool Bridge (CXR & Case Matching)
print("\n=== [B] SimilarCaseTool Bridge ===")
case_res = retriever.search_cases(
    query="deep sulcus sign tension pneumothorax",
    modality="CXR",
    k=2
)
for r in case_res:
    print(f"[{r.chunk.source_id}] {r.chunk.title} ({r.chunk.chunk_id}) | Score: {r.score:.4f}")

# 3. PatientHistoryTool / ReportGeneratorTool Bridge
print("\n=== [C] PatientHistoryTool / Education Bridge ===")
patient_res = retriever.search_patient_education(
    query="low sodium diet and daily weight monitoring for congestive heart failure",
    condition="Heart_Failure",
    k=2
)
for r in patient_res:
    print(f"[{r.chunk.source_id}] {r.chunk.title} ({r.chunk.chunk_id}) | Score: {r.score:.4f}")

=== [A] ClinicalGuidelineTool Bridge ===
[SRC_LOCAL_NICE_Pneumon] NICE Pneumonia Guideline (LOCAL_NICE_Pneumonia_Guideline#RECOMMENDATIONS_3) | Score: 0.0325
[SRC_LOCAL_NICE_Pneumon] NICE Pneumonia Guideline (LOCAL_NICE_Pneumonia_Guideline#TREATMENT_AND_SUPPORT_FOR_ONGOING_SYMPTOMS_6) | Score: 0.0318

=== [B] SimilarCaseTool Bridge ===
[OpenI_IU_CXR] OpenI Chest X-ray Report (CXR1019) (OpenI_CXR1019#FULL_REPORT_0) | Score: 0.0328
[OpenI_IU_CXR] OpenI Chest X-ray Report (CXR1021) (OpenI_CXR1021#FULL_REPORT_0) | Score: 0.0306

=== [C] PatientHistoryTool / Education Bridge ===
[SRC_LOCAL_living_with_] living with heart failure (LOCAL_living_with_heart_failure#MEDICATIONS_71) | Score: 0.0164
[SRC_LOCAL_living_with_] living with heart failure (LOCAL_living_with_heart_failure#MEDICATIONS_70) | Score: 0.0161


## 6. End-to-End Clinical Case Walkthrough
Simulating upstream Computer Vision findings and generating verified grounded evidence for LLM prompt synthesis.

In [7]:
patient_case = {
    "patient": "64yo Male",
    "chief_complaint": "Acute worsening shortness of breath, fever (38.5 C), and productive cough for 4 days.",
    "upstream_cv_findings": (
        "Chest X-Ray (PA view): Focal consolidation and airspace opacity in the right lower lobe "
        "with visible air bronchograms. Moderate blunting of the right costophrenic sulcus consistent "
        "with right-sided pleural effusion. Cardiothoracic ratio is mildly increased."
    ),
    "suspected_conditions": ["Pneumonia", "Pleural_Effusion", "Cardiomegaly"]
}

print("--- UPSTREAM CV MODEL INPUT ---")
print(f"Patient: {patient_case['patient']}")
print(f"CV Findings: {patient_case['upstream_cv_findings']}")
print(f"Suspected Conditions: {patient_case['suspected_conditions']}")

print("\n--- GROUNDED EVIDENCE SYNTHESIS (Ready for LLM Agent) ---")
grounded_context = retriever.get_grounded_context(
    query="What are the essential diagnostic findings, imaging confirmation, and management steps for pneumonia with pleural effusion?",
    k=3
)
print(grounded_context)

--- UPSTREAM CV MODEL INPUT ---
Patient: 64yo Male
CV Findings: Chest X-Ray (PA view): Focal consolidation and airspace opacity in the right lower lobe with visible air bronchograms. Moderate blunting of the right costophrenic sulcus consistent with right-sided pleural effusion. Cardiothoracic ratio is mildly increased.
Suspected Conditions: ['Pneumonia', 'Pleural_Effusion', 'Cardiomegaly']

--- GROUNDED EVIDENCE SYNTHESIS (Ready for LLM Agent) ---
--- EVIDENCE ITEM 1 ---
[SRC_LOCAL_Pleural_Effu] Pleural Effusion (Pleural Effusion | clinical_references | Section: MAIN | Audience: clinician)
Chunk ID: LOCAL_Pleural_Effusion#MAIN_31
Evidence Content:
[Pleural Effusion]
. Nonetheless, considering less common causes, such as trapped lung, is essential where pleural elastance measurements may be required. Urinothorax is another common cause of effusion, requiring pleural and serum creatinine levels for diagnosis.Treatment / ManagementManagement primarily focuses on identifying and treating 

## 7. Retrieval Evaluation Benchmark (Recall@K & MRR)
Automated benchmark test suite across 8 golden clinical queries.

In [8]:
evaluator = RetrievalEvaluator(retriever)
benchmark_results = evaluator.evaluate(k=3)

print(f"Recall@3: {benchmark_results['recall@3'] * 100}%")
print(f"Mean Reciprocal Rank (MRR): {benchmark_results['mrr']}")


Running retrieval benchmark across 8 golden test queries (k=3)...

  [HIT (Rank 1)] Q01_PNEUMONIA_CXR -> Consolidation
  [HIT (Rank 1)] Q02_PLEURAL_EFFUSION_DX -> Pleural Effusion
  [HIT (Rank 1)] Q03_PNEUMOTHORAX_SIGNS -> basic interpretation
  [HIT (Rank 1)] Q04_HF_GUIDELINES -> Pulmonary Edema
  [HIT (Rank 1)] Q05_STROKE_EMERGENCY -> MedlinePlus Stroke
  [HIT (Rank 1)] Q06_APPENDICITIS_IMAGING -> Case Report Appendicitis PMC13460692
  [HIT (Rank 1)] Q07_HF_PATIENT_CARE -> Heart Failure Diet
  [HIT (Rank 1)] Q08_PULMONARY_NODULE_RISK -> Consolidation

  Retrieval Evaluation Benchmark Summary
  Total Queries: 8
  Recall@3:     100.0%
  MRR:          1.000

Recall@3: 100.0%
Mean Reciprocal Rank (MRR): 1.0
